# Forget-MI tái lập + CE-selector gold-free · MIMIC 3/6/10% + IU 3%

Chạy Forget-MI (trung thành code gốc) với **selector 4-cách gold-free (S1–S4) INLINE**:
mỗi epoch tính CE (D_f vs D_nm_val) + snapshot ứng viên; cuối cùng eval selected trên D_t_final.
KHÔNG lưu 30 checkpoint, KHÔNG dùng GOLD/test-cuối để chọn. **KHÔNG đổi training/loss.**

**Cell 2**: chọn `DATASET` (mimic/iu) + `FORGET_PCT`. Mỗi Save Version = 1 cấu hình (~1.1h).


In [ ]:
# Cell 1: setup
import os, subprocess
WORK='/kaggle/working'; REPO=f'{WORK}/Forget-MI-LoKU'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/nhnhu146/Forget-MI-LoKU.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
os.chdir(REPO)
assert os.path.exists('training/ce_selector_pilot.py'),'push code truoc + re-import notebook'
subprocess.run(['pip','install','-q','pydicom','scikit-image','scikit-learn','pyyaml','wandb','seaborn==0.13.2'],check=True)
subprocess.run(['pip','install','-q','transformers==4.38.0','peft==0.10.0','accelerate==0.27.0'],check=True)
import torch; assert torch.cuda.is_available(),'Bat GPU'
print('Commit:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())
print('GPU   :',torch.cuda.get_device_name(0))


In [ ]:
# Cell 2: CHON + path discovery (mimic/iu)
import glob, os
DATASET    = 'mimic'   # 'mimic' | 'iu'
FORGET_PCT = 3         # 3 | 6 | 10  (iu: 3)
SEED       = 42
assert DATASET in ('mimic','iu') and FORGET_PCT in (3,6,10)
if DATASET=='iu': assert FORGET_PCT==3,'IU chi co 3%'

def fd(*slugs):
    for s in slugs:
        if os.path.isdir(f'/kaggle/input/{s}'): return f'/kaggle/input/{s}'
        h=glob.glob(f'/kaggle/input/datasets/*/{s}')
        if h: return sorted(h)[0]
    return None
def first_existing(root, rels):
    for r in rels:
        p=os.path.join(root,r)
        if os.path.exists(p): return p
    return None
def bins(root): return sorted(glob.glob(os.path.join(root,'**','pytorch_model.bin'),recursive=True),key=len)

tag=f'{DATASET}{FORGET_PCT}per'
RUN_ID    =f'forgetmi_{tag}_s{SEED}'
OUTPUT_DIR=f'/kaggle/working/fmi_output/{tag}_s{SEED}'
SEL_DIR   =f'/kaggle/working/checkpoint_selection_{tag}_s{SEED}'

if DATASET=='mimic':
    CONFIG='config_baseline_kaggle.yaml'
    DATA=fd('forget-mi-data'); MOD=fd('forget-mi-models-full','forget-mi-models')
    assert DATA and MOD,'Add forget-mi-data + forget-mi-models-full'
    BASE=os.path.dirname([b for b in bins(MOD) if 'training_original_model' in b][0])
    gh=[b for b in bins(MOD) if f'model_retrained_{FORGET_PCT}per' in b]
    GOLD=os.path.dirname(gh[0]) if gh else BASE
    TEXT=first_existing(DATA,['data/metadata','metadata']); IMG=first_existing(DATA,['data/img_data','img_data'])
    SPLIT='./data_splits/mimic-cxr-sub-img-edema-split-manualtest.csv'
    FORGET=f'./data_splits/forget_set_{FORGET_PCT}per.csv'
else:
    CONFIG='config_baseline_iu_kaggle.yaml'
    DATA=fd('forget-mi-data-iu'); MOD=fd('forget-mi-models-iu'); MODRE=fd('forget-mi-models-iu-re')
    RAD=fd('chest-xrays-indiana-university')
    assert DATA and MOD and RAD,'Add forget-mi-data-iu + forget-mi-models-iu + forget-mi-models-iu-re + chest-xrays-indiana-university'
    ogb=[b for b in bins(MOD) if 'model_og' in b.lower() or 'base_model' in b.lower()] or bins(MOD)
    BASE=os.path.dirname(ogb[0])
    reb=(bins(MODRE) if MODRE else []) or [b for b in bins(MOD) if 'retrain' in b.lower()]
    GOLD=os.path.dirname(reb[0]) if reb else BASE
    # TEXT: dir chua all_data.tsv (glob DATA roi toan bo input); IMG: anh .png (data/img_data hoac raddar)
    tsv=glob.glob(os.path.join(DATA,'**','all_data.tsv'),recursive=True) or glob.glob('/kaggle/input/**/all_data.tsv',recursive=True)
    TEXT=os.path.dirname(tsv[0]) if tsv else first_existing(DATA,['data/metadata','metadata'])
    IMG=first_existing(DATA,['data/img_data','img_data']) or (first_existing(RAD,['images/images_normalized','images']) if RAD else None) or RAD
    sp=glob.glob(os.path.join(DATA,'**','iu-split.csv'),recursive=True) or glob.glob('/kaggle/input/**/iu-split.csv',recursive=True) or glob.glob(os.path.join(DATA,'**','*iu*split*.csv'),recursive=True)
    fg=glob.glob(os.path.join(DATA,'**',f'forget_set_{FORGET_PCT}per_iu.csv'),recursive=True) or glob.glob(f'/kaggle/input/**/forget_set_{FORGET_PCT}per_iu.csv',recursive=True)
    assert sp and fg,f'Khong thay iu-split / forget_set_iu (glob toan input)'
    SPLIT=sp[0]; FORGET=fg[0]
    if not TEXT or not IMG:                     # DIAGNOSTIC: in cay input de do path
        print('⚠️ TEXT',TEXT,'IMG',IMG,'- liet ke input:')
        for r,d,f in os.walk(DATA):
            if r[len(DATA):].count(os.sep)<=2: print(' ',r,'->',[x for x in f][:4])

for n,p in {'base':BASE,'text':TEXT,'img':IMG,'split':SPLIT,'forget':FORGET}.items():
    assert p and os.path.exists(p),f'Missing {n}: {p}'
COMMON_OVR={'forget_set_path':FORGET,'base_model_path':BASE,'bert_pretrained_dir':BASE,
            'retrained_model_path':GOLD,'text_data_dir':TEXT,'img_data_dir':IMG,'data_split_path':SPLIT}
print('DATASET',DATASET,'PCT',FORGET_PCT,'| config',CONFIG,'| tag',tag)
print('BASE',BASE); print('SPLIT',SPLIT); print('SEL_DIR',SEL_DIR)


In [ ]:
# Cell 3: TRAIN Forget-MI + CE-SELECTOR INLINE (1 luot, ~1.1h)
#   ce_selector_out=SEL_DIR -> moi epoch tinh CE + snapshot ung vien; evaluate_last_and_best=1
#   (khong luu 30 ckpt); eval_every_epoch=0. KHONG doi training/loss.
import os, subprocess, time
ovr=dict(COMMON_OVR); ovr.update({'output_dir':OUTPUT_DIR,
    'results_csv_path':'/kaggle/working/results_fmi_native.csv',
    'evaluate_last_and_best':1, 'eval_every_epoch':0, 'id':RUN_ID,
    'ce_selector_out':SEL_DIR, 's4_delta':0.15})
arg=','.join(f'{k}={v}' for k,v in ovr.items())
env={**os.environ,'PYTHONPATH':'.','WANDB_MODE':'disabled'}
cmd=['python','training/forgetmi_partial.py','--config',CONFIG,'--seed',str(SEED),'--fresh','--override',arg]
print('='*70); print('TRAIN + INLINE CE-SELECTOR',RUN_ID); print('='*70)
t0=time.time()
try:
    subprocess.run(cmd,env=env,check=True); print(f'DONE ({(time.time()-t0)/3600:.2f}h)')
except subprocess.CalledProcessError as e:
    print(f'FAIL rc={e.returncode}')


In [ ]:
# Cell 4: kiem tra output selector
import os, glob
print('SEL_DIR:',SEL_DIR)
print('Files  :',sorted(os.path.basename(p) for p in glob.glob(f'{SEL_DIR}/*')))
assert os.path.exists(os.path.join(SEL_DIR,'selected_checkpoints.json')),\
    'Khong thay selected_checkpoints.json — xem log Cell 3 (dong [split] va 🧭 selector)'


In [ ]:
# Cell 5: xem ket qua 4 selector
import os, json, pandas as pd
pd.set_option('display.width',180)
traj=os.path.join(SEL_DIR,'forgetmi_selector.csv')
if os.path.exists(traj):
    print('===== TRAJECTORY (gold-free) ====='); print(pd.read_csv(traj).to_string(index=False))
sj=os.path.join(SEL_DIR,'selected_checkpoints.json')
if os.path.exists(sj):
    d=json.load(open(sj)); res=d['results']
    print(f"\n===== {tag}: 4 CACH CHON (s4_delta={d.get('s4_delta')}) =====")
    tab=[]
    for k,v in res.items():
        if v.get('epoch') is None or 'Df_AUC' not in v:
            tab.append({'selector':k,'epoch':(f"E{v.get('epoch')}" if v.get('epoch') is not None else 'NO CROSSING')})
        else:
            tab.append({'selector':k,'epoch':f"E{v['epoch']}",'Df_AUC':v['Df_AUC'],'Df_F1':v['Df_F1'],
                        'Dt_AUC':v['Dt_AUC'],'Dt_F1':v['Dt_F1'],'MIA':v['MIA']})
    print(pd.DataFrame(tab).to_string(index=False))
    uniq=sorted(set(v['epoch'] for v in res.values() if v.get('epoch') is not None))
    print(f"\n-> {len(uniq)} epoch khac nhau: {['E'+str(e) for e in uniq]}  ({'DONG THUAN' if len(uniq)<=2 else 'phan tan'})")
print('\nOutput:',SEL_DIR)
